**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** is an unsupervised machine learning clustering algorithm.It looks at the density of data points in space, grouping tight clusters together and automatically identifying stray data points as outliers (noise).

# The Core Intuition

Instead of guessing a center, you choose a search circle size (**Epsilon**) and a minimum number of points required to form a crowd (**MinSamples**).
1. You pick a random point and draw a circle around it.
2. If that circle captures enough points to meet your crowd requirement, those points form a tiny cluster.
3. You then move to those newly discovered points and draw circles around them too.
4. As long as these dense circles keep overlapping and touching each other, the cluster expands and snakes forward like a chain reaction.
5. This specific chain-link behavior allows it to perfectly wrap around complex shapes like **crescent moons or rings**. If a circle contains too few points and doesn't touch any crowd, it is thrown out as an outlier.



# Mathematical Example

1. **Dataset**

    |Point | Coordinate ($x_1 , x_2$)|
    |---|---|
    |A|(1,1)|
    |B|(2,2)|
    |C|(2,1)|
    |D|(3,2)|
    |E|(6,6)|

2. **Setup & Hyperparameters**

    - Epsilon ($\epsilon $) = $sqrt{2} \approx 1.41$
    - MinSamples = 3 (A circle must contain at least 3 points, including itself, to be considered dense).

3. **Evaluate a Random Point (Point A)**
    DBSCAN selects a random unvisited coordinate, Let say Point A (1,1), and calculates the Euclidean distance to its neighbors to see who lands inside its radius of 1.41.

    - Distance to B: $\sqrt{(1-2)^2 + (1-2)^2} = \sqrt{1 + 1} = \mathbf{1.41}$ (Inside!)
    - Distance to C: $\sqrt{(1-2)^2 + (1-1)^2} = \sqrt{1 + 0} = \mathbf{1.00}$ (Inside!)
    - Distance to D: $\sqrt{(1-3)^2 + (1-2)^2} = \sqrt{4 + 1} = 2.23$ (Outside)
    - Distance to E: $\sqrt{(1-6)^2 + (1-6)^2} = \sqrt{25 + 25} = 7.07$ (Outside)
    The Count: Points inside A's circle are {A, B, C}. Total = 3.

    **Mathematical Classification**
    Because $3 \ge \text{MinSamples}$, Point A is mathematically declared a Core Point. A brand-new cluster (Cluster 1) is successfully born, containing points A, B, and C.

4. **Expand the Chain Reaction**
    
    **(Evaluate Point B)**

    DBSCAN must now check the neighbors it just discovered to see if the chain reaction continues. It moves to Point B (2,2) and draws a circle.

    - Distance to A: $\mathbf{1.41}$ (Inside)
    - Distance to C: $\sqrt{(2-2)^2 + (2-1)^2} = \sqrt{0 + 1} = \mathbf{1.00}$ (Inside)
    - Distance to D: $\sqrt{(2-3)^2 + (2-2)^2} = \sqrt{1 + 0} = \mathbf{1.00}$ (Inside!)
    - Distance to E: $\sqrt{(2-6)^2 + (2-6)^2} = \sqrt{16 + 16} = 5.65$ (Outside)
    
    The Count: Points inside B's circle are {B, A, C, D}. Total = 4.
    
    **Mathematical Classification**
    Because $4 \ge \text{MinSamples}$, Point B is also a Core Point. Point B expands the cluster, and its new neighbor Point D is pulled into Cluster 1. 

    **Evaluate Point C**

    Calculate distances from C to all other points:
    - Distance to A: $\sqrt{(2-1)^2 + (1-1)^2} = \mathbf{1.00}$ (Inside)
    - Distance to B: $\sqrt{(2-2)^2 + (1-2)^2} = \mathbf{1.00}$ (Inside)
    - Distance to D: $\sqrt{(2-3)^2 + (1-2)^2} = \sqrt{1 + 1} = \mathbf{1.41}$ (Inside)
    - Distance to E: $\sqrt{(2-6)^2 + (1-6)^2} = \sqrt{16 + 25} = 6.40$ (Outside)

    **Mathematical Classification**

    Point C is also a Core Point.

    **Evaluate Point D**

    Calculate distances from D to all other points:
    - Distance to B: $\mathbf{1.00}$ (Inside)
    - Distance to C: $\mathbf{1.41}$ (Inside)
    - Distance to A: $2.23$ (Outside)
    - Distance to E: $\sqrt{(3-6)^2 + (2-6)^2} = \sqrt{9 + 16} = \mathbf{5.00}$ (Outside)

    **Mathematical Classification**

    Point D is also a Core Point.


5. **Evaluate Remaining Unvisited Points (Point E)**
    
    DBSCAN now moves to the next unvisited point in the dataset: Point E (6,6).
    Distance to all other points (A, B, C, D) is $\ge 5.00$ (All Outside radius 1.41).

    The Count: The only point inside E's circle is itself. Total = 1.

    **Mathematical Classification**

    Because $1 < \text{MinSamples}$ and it does not touch any active core circles, Point E is mathematically classified as Noise (an Outlier). It is completely isolated and denied entry into any cluster


**Summary**

|Point|Coordinate|Neighborhood Count|Role|Cluster Assignment|
|---|---|---|---|---|
|A|$(1,1)$|3|Core Point|Cluster 1|
|B|$(2,2)$|4|Core Point|Cluster 1|
|C|$(2,1)$|4|Core Point|Cluster 1|
|D|$(3,2)$|3|Core Point|Cluster 1|
|E|$(6,6)$|1|Noise Point|-1 (Outlier)|

**Final Cluster Output**
- Cluster 1: {A, B, C, D}
- Noise: {E}


# Essential Terminologies You Must Know

- Epsilon ($\epsilon$): The maximum distance radius that defines a point's local neighborhood.
- MinPts ($MinSamples$): The minimum number of points required within an $\epsilon$-neighborhood to form a dense region.
- Core Point: A point with at least $MinPts$ within its $\epsilon$-neighborhood.
- Border Point: A point that has fewer than $MinPts$ within its $\epsilon$-neighborhood, but falls inside the $\epsilon$-neighborhood of a Core Point.
- Noise Point: Any point that is neither a Core Point nor a Border Point.
- Directly Density-Reachable: A point $P$ is directly density-reachable from point $Q$ if $P \in N_{\epsilon}(Q)$ and $Q$ is a Core Point.
- Density-Connected: Two points $P$ and $Q$ are density-connected if there exists an intermediate point $O$ such that both $P$ and $Q$ are density-reachable from $O$.

# Python Implementation

In [ ]:
import numpy as np


class DBSCAN:

  def __init__(self, eps=1.4142, min_samples=3):
    self.eps = eps
    self.min_samples = min_samples
    self.labels_ = None

  def _region_query(self, X, point_idx):
    """Calculates Euclidean distances and returns indices of all points within eps radius."""
    distances = np.linalg.norm(X - X[point_idx], axis=1)
    return np.where(distances <= self.eps)[0]

  def fit_predict(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples = X.shape[0]

    # 0 = Unvisited, -1 = Noise/Outlier, >0 = Cluster ID
    self.labels_ = np.zeros(n_samples, dtype=int)
    cluster_id = 0

    for i in range(n_samples):
      # Skip if point has already been processed
      if self.labels_[i] != 0:
        continue

      # Find neighbors within eps radius
      neighbors = self._region_query(X, i)

      # Check if point satisfies Core Point criteria
      if len(neighbors) < self.min_samples:
        self.labels_[i] = -1  # Mark as Noise
      else:
        cluster_id += 1
        self.labels_[i] = cluster_id

        # Expand cluster using seed queue
        seed_queue = list(neighbors)

        while len(seed_queue) > 0:
          current_point = seed_queue.pop(0)

          # Reassign Noise point to Border point of current cluster
          if self.labels_[current_point] == -1:
            self.labels_[current_point] = cluster_id

          # Skip if point is already assigned to a cluster
          if self.labels_[current_point] != 0:
            continue

          # Mark current point as part of the cluster
          self.labels_[current_point] = cluster_id

          # Find neighbors of current point to continue chain reaction
          current_neighbors = self._region_query(X, current_point)

          if len(current_neighbors) >= self.min_samples:
            seed_queue.extend(current_neighbors)

    return self.labels_


# TEST SCRIPT: 2D DATASET EXAMPLE

# Points: A(1,1), B(2,2), C(2,1), D(3,2), E(6,6)
X_train = np.array([[1, 1], [2, 2], [2, 1], [3, 2], [6, 6]])
point_names = ["A", "B", "C", "D", "E"]

# Hyperparameters: eps = sqrt(2) ≈ 1.4142, MinSamples = 3
dbscan = DBSCAN(eps=1.4142, min_samples=3)
labels = dbscan.fit_predict(X_train)

print("=== DBSCAN EXECUTION RESULTS ===")

for i, name in enumerate(point_names):
  coord = X_train[i]
  status = f"Cluster {labels[i]}" if labels[i] != -1 else "Noise (-1)"
  print(f"Point {name} {coord} -> {status}")

=== DBSCAN EXECUTION RESULTS ===
Point A [1 1] -> Cluster 1
Point B [2 2] -> Cluster 1
Point C [2 1] -> Cluster 1
Point D [3 2] -> Cluster 1
Point E [6 6] -> Noise (-1)
